# CineMovie — Recommendation Model

## Objective

Build a content-based movie recommendation engine using the cleaned `new_movie_df` produced during EDA.

Pipeline:

1. Prepare movie tags
2. Convert tags into TF-IDF vectors
3. Calculate cosine similarity
4. Recommend the most similar movies
5. Test successful and failure scenarios

> **Baseline:** TF-IDF + cosine similarity. Bag-of-Words is kept as a future comparison experiment.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Input Data

The recommendation model uses `new_movie_df`, containing:

- `movie_id`
- `title`
- `tags`

The `tags` field is the combined textual representation of each movie's overview, genres, keywords, cast, and crew/director information.

Run the completed EDA notebook first if `new_movie_df` is not already available in the current Jupyter session.

In [ ]:
# Confirm that the cleaned recommendation dataframe is available.
new_movie_df[['movie_id', 'title', 'tags']].head()

In [ ]:
# Check the number of movies and columns used by the recommendation model.
new_movie_df.shape

## 2. TF-IDF Vectorization

TF-IDF converts the combined movie tags into numerical vectors.

`stop_words='english'` removes common English stop words. It does not merge words such as `actor` and `actors`; those remain separate tokens in the baseline model.

`max_features=5000` limits the vocabulary to the 5,000 most useful features.

In [ ]:
# Create the TF-IDF vectorizer.
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

In [ ]:
# Transform the movie tags into TF-IDF vectors.
vectors = tfidf.fit_transform(new_movie_df['tags'])

In [ ]:
# Check the shape of the TF-IDF matrix.
vectors.shape

In [ ]:
# Display the first 100 terms learned by the TF-IDF vectorizer.
tfidf.get_feature_names_out()[:100]

In [ ]:
# Display the total number of features learned by the vectorizer.
len(tfidf.get_feature_names_out())

## 3. Cosine Similarity

Cosine similarity measures the similarity between the TF-IDF representation of every pair of movies.

The resulting matrix has one row and one column for every movie.

In [ ]:
# Calculate cosine similarity between every pair of movies.
similarity = cosine_similarity(vectors)

In [ ]:
# Check the shape of the cosine similarity matrix.
similarity.shape

In [ ]:
# Inspect a small portion of the similarity matrix.
similarity[:5, :5]

## 4. Recommendation Function

The function:

1. Finds the selected movie.
2. Retrieves its similarity scores.
3. Sorts movies by similarity.
4. Removes the selected movie itself.
5. Returns the requested number of recommendations.

The function also handles an invalid movie title without producing a traceback.

In [ ]:
def recommend(movie_title, top_n=10):
    # Check whether the requested movie exists in the dataset.
    if movie_title not in new_movie_df['title'].values:
        return f"Movie '{movie_title}' was not found in the dataset."

    # Find the index of the selected movie.
    movie_index = new_movie_df[new_movie_df['title'] == movie_title].index[0]

    # Get similarity scores for the selected movie.
    movie_similarity = similarity[movie_index]

    # Sort movie indices by similarity in descending order.
    sorted_indices = sorted(
        enumerate(movie_similarity),
        key=lambda x: x[1],
        reverse=True
    )

    # Store the recommendations.
    recommendations = []

    # Skip the selected movie itself and return top_n results.
    for index, score in sorted_indices[1:top_n + 1]:
        recommendations.append({
            'title': new_movie_df.iloc[index]['title'],
            'similarity_score': round(score, 4)
        })

    return pd.DataFrame(recommendations)

## 5. Manual Recommendation Checks

The baseline was tested on different movie types to assess whether the recommendations are sensible and to identify weaknesses.

Successful cases include `Avatar`, `The Dark Knight Rises`, `Titanic`, `Toy Story`, and `Inception`.

Failure cases include an invalid title and empty input.

In [ ]:
# Test the recommendation system with Avatar.
avatar_recommendations = recommend('Avatar', top_n=5)
avatar_recommendations

In [ ]:
# Test the recommendation system with The Dark Knight Rises.
dark_knight_recommendations = recommend('The Dark Knight Rises', top_n=5)
dark_knight_recommendations

In [ ]:
# Test the recommendation system with Titanic.
titanic_recommendations = recommend('Titanic', top_n=5)
titanic_recommendations

In [ ]:
# Test the recommendation system with Toy Story.
toy_story_recommendations = recommend('Toy Story', top_n=5)
toy_story_recommendations

In [ ]:
# Test the recommendation system with Inception.
inception_recommendations = recommend('Inception', top_n=5)
inception_recommendations

In [ ]:
# Failure scenario: invalid movie title.
invalid_recommendation = recommend('Avtar', top_n=5)
invalid_recommendation

In [ ]:
# Failure scenario: empty movie title.
empty_recommendation = recommend('', top_n=5)
empty_recommendation

## Baseline Findings

The baseline demonstrates strong content matching in some cases, such as `The Dark Knight Rises` and `Toy Story`, where closely related franchise movies rank highly.

Other cases, such as `Titanic` and `Inception`, show that lexical TF-IDF similarity can emphasize shared words or metadata without fully capturing human semantic similarity.

These observations will be carried forward into the evaluation and limitations sections.

## Future Improvements

- Compare TF-IDF with Bag-of-Words as an alternative text representation.
- Experiment with stemming or lemmatization.
- Investigate semantic embeddings for better meaning-level similarity.
- Explore field weighting so genres/keywords/cast/crew do not contribute equally by default.
- Add diversity-aware re-ranking.
- Consider hybrid recommendation using collaborative/user-preference signals when interaction data becomes available.